In [1]:
from dataclasses import dataclass, field
import torch


@dataclass
class TrainConfig:
    model_name: str = "Qwen/Qwen3-0.6B"
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")

    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: list[str] = field(default_factory=lambda: ["q_proj", "v_proj"])
    bias: str = "none"
    task_type: str = "CAUSAL_LM"

    num_train_epochs: int = 25
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    learning_rate: float = 1e-4
    logging_steps: int = 10
    eval_strategy: str = "steps"
    eval_steps: int = 50
    save_steps: int = 50
    save_total_limit: int = 2
    fp16: bool = True
    gradient_accumulation_steps: int = 4

    dataset_name: str = "banking77"
    max_length: int = 128
    val_split: float = 0.1

    output_dir: str = "./qwen3_lora"
    log_file: str = "./validation_results.txt"
    push_to_hub: bool = False
    hub_model_id: str = ""
    hub_token: str = ""

    num_val_samples: int = 5

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, DataCollatorForLanguageModeling, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import torch

# loading model
def load_model(config: TrainConfig):
    """
    Loads model and tokenizer from pretrained
    """
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype="auto",
        device_map="auto"
    )

    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.target_modules,
        lora_dropout=config.lora_dropout,
        bias=config.bias,
        task_type=config.task_type
    )

    lora = get_peft_model(model, lora_config)

    return tokenizer, lora

# preprocessing and loading dataset
def make_preprocess(tokenizer, max_length):
    def preprocess(batch):
        texts = [
            f"Input: {inp}\nTarget: {tgt}"
            for inp, tgt in zip(batch["input"], batch["target"])
        ]
        tokenized = tokenizer(texts, max_length=max_length, truncation=True, padding="max_length")
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized
    return preprocess

def create_dataset(config: TrainConfig, tokenizer):
    dataset = load_dataset("zamal/github-meta-data")
    split = dataset["train"].train_test_split(test_size=config.val_split, seed=42)

    tokenized = split.map(make_preprocess(tokenizer, config.max_length), batched=True, remove_columns=["input", "target"])
    tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return split, tokenized

# training LoRA and logging the results
def train_and_log(tokenizer, model, dataset, config: TrainConfig):
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=config.output_dir,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        learning_rate=config.learning_rate,
        num_train_epochs=config.num_train_epochs,
        logging_steps=config.logging_steps,
        eval_strategy=config.eval_strategy,
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        save_total_limit=config.save_total_limit,
        fp16=config.fp16,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        push_to_hub=config.push_to_hub,
        hub_model_id=config.hub_model_id if config.push_to_hub else None,
        hub_token=config.hub_token if config.push_to_hub else None,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        data_collator=data_collator
    )

    trainer.train()

    model.save_pretrained(config.output_dir)
    tokenizer.save_pretrained(config.output_dir)

    return model

# checking if model is ok
def val_check(tokenizer, model, dataset, config: TrainConfig):
    model.eval()
    lines = []

    for i in range(config.num_val_samples):
        inp = dataset[i]["input"]
        target = dataset[i]["target"]

        prompt = f"Input: {inp}\nTarget:"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=config.max_length).to(model.device)

        with torch.no_grad():
            output = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=64,
            )

        prediction = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        lines.append(f"[{i+1}]\nInput:      {inp}\nTarget:     {target}\nPrediction: {prediction}\n\n")

    with open(config.log_file, "w") as f:
        f.writelines(lines)

# main function
def run_lora_pipeline(config: TrainConfig):
    tokenizer, model = load_model(config)
    print("Модель подгрузили")
    raw, dataset = create_dataset(config, tokenizer)
    print("Датасет подгрузили")
    print("Начинаем обучать")
    trained = train_and_log(tokenizer, model, dataset, config)
    print("Обучили, проверяем")
    val_check(tokenizer, trained, raw["test"], config)
    print("Проверили")
    return f"Модель {config.model_name} обучена и загружена в {config.output_dir}"

In [3]:
config = TrainConfig()
run_lora_pipeline(config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Датасет подгрузили
Начинаем обучать


Step,Training Loss,Validation Loss
50,2.076405,2.132153
100,1.741769,1.910762
150,1.535720,1.814676
200,1.308037,1.761358
250,1.172092,1.718832
300,1.062504,1.732035
350,0.982702,1.738173
400,0.933260,1.749837
450,0.907717,1.760826


Обучили, проверяем
Проверили


'Модель Qwen/Qwen3-0.6B обучена и загружена в ./qwen3_lora'